# Design inspection: what is stored in one `.oas` file

Opens a single `.oas` file and shows its shapes **as stored**, before any contact centre is
computed. Use it to check how one contact is built from several shapes (fractures), whether those
shapes overlap, touch or repeat, and what centre merging them would give.

| Step | Function | What it shows |
|---|---|---|
| File overview | `load_layout`, `list_layers`, `list_shapes` | database unit, top cell, shapes per layer |
| Shape table | `list_shapes` | every stored shape: kind, source cell, bounding box, area, vertices |
| Duplicates | `list_shapes` | shapes stored more than once at the same place |
| Merged patterns | `merge_shapes` | union of overlapping/touching shapes, one centroid per pattern |
| Plots | matplotlib | whole file, and one pattern in detail |

Coordinates: tile-local frame, (0, 0) = file centre, y up. Positions in µm, sizes and
differences in nm. Set the path below and run all cells.

In [ ]:
OAS_PATH = r"path\to\tile.oas"
LAYER = None              # (layer, datatype) of the contacts; None = the only layer in the file
CELL = None               # top cell name; None if the file has a single top cell
ZOOM_AT_UM = (0.0, 0.0)   # tile-local position (µm); the pattern nearest to it is shown in detail

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.collections import PolyCollection

from affine_ransac.geometry.polygon import polygon_area, polygon_centroid
from affine_ransac.io.design import get_top_cell, list_layers, list_shapes, load_layout, merge_shapes

## 1. File overview

In [ ]:
layout = load_layout(OAS_PATH)
print(f"Database unit: {layout.dbu * 1000:g} nm")
print(f"Top cell: {get_top_cell(layout, CELL).name}")

for layer in list_layers(layout):
    kinds = pd.Series([s["kind"] for s in list_shapes(layout, *layer, CELL)], dtype=object)
    print(f"Layer {layer}: {len(kinds)} shapes {kinds.value_counts().to_dict()}")

if LAYER is None:
    (LAYER,) = list_layers(layout)  # fails if the file has several layers: then set LAYER
print(f"Inspecting layer {LAYER}")

## 2. Shape table

One row per stored shape (texts skipped, hierarchy flattened). `cell` is the cell the shape is
defined in: several cells or repeated placements point to shapes coming from the hierarchy.

In [ ]:
shapes = list_shapes(layout, *LAYER, CELL)
lo = np.array([s["vertices"].min(axis=0) for s in shapes])
hi = np.array([s["vertices"].max(axis=0) for s in shapes])
centroids = np.array([polygon_centroid(s["vertices"]) for s in shapes])

shape_table = pd.DataFrame({
    "kind": [s["kind"] for s in shapes],
    "cell": [s["cell"] for s in shapes],
    "x_um": centroids[:, 0] / 1000, "y_um": centroids[:, 1] / 1000,
    "w_nm": hi[:, 0] - lo[:, 0], "h_nm": hi[:, 1] - lo[:, 1],
    "area_nm2": [s["area_nm2"] for s in shapes],
    "n_vertices": [len(s["vertices"]) for s in shapes],
})
print(f"{len(shape_table)} shapes")
print("By kind:", shape_table.kind.value_counts().to_dict())
print("By cell:", shape_table.cell.value_counts().to_dict())
print("Most common sizes (w_nm, h_nm):")
print(shape_table.groupby(["w_nm", "h_nm"]).size().sort_values(ascending=False).head(10).to_string())
shape_table

## 3. Duplicates

Shapes with exactly the same vertices, stored more than once.

In [ ]:
keys = pd.Series([tuple(np.round(s["vertices"], 6).ravel()) for s in shapes])
copies = keys.map(keys.value_counts())
print(f"{(copies > 1).sum()} of {len(shapes)} shapes have an exact duplicate "
      f"({keys[copies > 1].nunique()} distinct shapes repeated)")
shape_table[copies > 1].assign(copies=copies[copies > 1]).sort_values(["x_um", "y_um"])

## 4. Merged patterns

`merge_shapes` joins shapes that overlap or touch into one pattern (KLayout boolean OR) and tells
which pattern each shape belongs to. Per pattern:

- `n_shapes`: shapes it is built from; `overlap_nm2` = sum of shape areas − pattern area (> 0 means
  the shapes overlap);
- `max_offset_nm`: largest distance of a shape centroid from the pattern centroid, i.e. how far the
  current per-shape centres are from the merged centre.

The pattern area and centroid use the outer outline only (a hole in a pattern would be ignored).

In [ ]:
merged, labels = merge_shapes(layout, *LAYER, CELL)
print(f"{len(shapes)} shapes -> {len(merged)} merged patterns; {np.sum(labels < 0)} shapes without area")

pattern_xy = np.array([polygon_centroid(m) for m in merged])
pattern_lo = np.array([m.min(axis=0) for m in merged])
pattern_hi = np.array([m.max(axis=0) for m in merged])
pattern_area = np.array([polygon_area(m) for m in merged])
ok = labels >= 0
shapes_area = np.bincount(labels[ok], weights=shape_table.area_nm2[ok], minlength=len(merged))
offsets = np.linalg.norm(centroids[ok] - pattern_xy[labels[ok]], axis=1)
max_offset = np.zeros(len(merged))
np.maximum.at(max_offset, labels[ok], offsets)

patterns = pd.DataFrame({
    "x_um": pattern_xy[:, 0] / 1000, "y_um": pattern_xy[:, 1] / 1000,
    "w_nm": pattern_hi[:, 0] - pattern_lo[:, 0], "h_nm": pattern_hi[:, 1] - pattern_lo[:, 1],
    "n_shapes": np.bincount(labels[ok], minlength=len(merged)),
    "area_nm2": pattern_area, "overlap_nm2": shapes_area - pattern_area,
    "max_offset_nm": max_offset,
})
print("Patterns by number of shapes:", patterns.n_shapes.value_counts().sort_index().to_dict())
print(f"Patterns with overlapping shapes: {(patterns.overlap_nm2 > 1e-6).sum()}")
print("Most common pattern sizes (w_nm, h_nm):")
print(patterns.groupby(["w_nm", "h_nm"]).size().sort_values(ascending=False).head(10).to_string())
patterns

## 5. Whole file

Stored shapes are filled half-transparent, so overlaps look darker. Merged outlines in red, merged
centroids as `+`. Zoom with the toolbar.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.add_collection(PolyCollection([s["vertices"] / 1000 for s in shapes],
                                 facecolors="tab:blue", edgecolors="tab:blue", alpha=0.25, linewidths=0.5))
ax.add_collection(PolyCollection([m / 1000 for m in merged], facecolors="none", edgecolors="red", linewidths=0.8))
ax.plot(patterns.x_um, patterns.y_um, "k+", markersize=5)
ax.autoscale_view()
ax.set_aspect("equal")
ax.set_xlabel("x (µm)")
ax.set_ylabel("y (µm)")
ax.set_title(f"{len(shapes)} stored shapes, {len(merged)} merged patterns")
plt.show()

## 6. One pattern in detail

The pattern nearest to `ZOOM_AT_UM`: each of its shapes in its own colour, labelled with its row
in the shape table. Axes are nm relative to the merged centroid; `x` = shape centroids (what the
current reader returns per shape), `+` = merged centroid. Change `ZOOM_AT_UM` and re-run this
cell to look at another pattern.

In [ ]:
j = int(np.argmin(np.linalg.norm(pattern_xy / 1000 - np.array(ZOOM_AT_UM), axis=1)))
members = np.flatnonzero(labels == j)
origin = pattern_xy[j]
print(f"Pattern {j} at ({patterns.x_um[j]:.4f}, {patterns.y_um[j]:.4f}) µm, built from {len(members)} shapes")
print(patterns.loc[j].to_string())

fig, ax = plt.subplots(figsize=(7, 7))
for k, i in enumerate(members):
    v = shapes[i]["vertices"] - origin
    color = f"C{k % 10}"
    ax.fill(v[:, 0], v[:, 1], color=color, alpha=0.25)
    ax.plot(*np.vstack([v, v[:1]]).T, color=color, linewidth=1)
    c = centroids[i] - origin
    ax.plot(*c, "x", color=color)
    ax.annotate(str(i), c, textcoords="offset points", xytext=(4, 4), color=color)
outline = merged[j] - origin
ax.plot(*np.vstack([outline, outline[:1]]).T, "k--", linewidth=1, label="merged outline")
ax.plot(0, 0, "k+", markersize=12, label="merged centroid")
ax.set_aspect("equal")
ax.set_xlabel("x − x₀ (nm)")
ax.set_ylabel("y − y₀ (nm)")
ax.legend()
plt.show()

shape_table.loc[members].assign(
    dx_nm=centroids[members, 0] - origin[0], dy_nm=centroids[members, 1] - origin[1])

In [ ]:
# Vertices of each shape of this pattern, nm relative to the merged centroid.
for i in members:
    print(f"shape {i} ({shapes[i]['kind']}, cell {shapes[i]['cell']}):")
    print(np.round(shapes[i]["vertices"] - origin, 3))